# NI Tracking 2026

本 notebook 在 `ni_post_tracking` 分支下开发。

- **数据源**：Databricks (`bigu_ps_sandbox`) + 本地 CSV (`Data/`)
- **凭据**：`NI_Project_2026/.env`（不进 git）
- **集群自动唤醒**：连接前自动检测并启动 stopped 的 all-purpose cluster

## 1. Connect to Databricks (auto-start cluster)

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from databricks import sql

# -----------------------------------------------------------
# 1) 定位 .env（放在 repo 上一级的 NI_Project_2026/）
# -----------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / ".env").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / ".env").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = Path(
        r"C:\Users\wang.r.19\OneDrive - Procter and Gamble\Documents\NI_Project_2026"
    )
load_dotenv(PROJECT_ROOT / ".env", override=True)

# -----------------------------------------------------------
# 2) 解析配置
# -----------------------------------------------------------
server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME", "").replace("https://", "").strip("/")
http_path       = os.getenv("DATABRICKS_HTTP_PATH", "").strip()
if not http_path.startswith("/"):
    http_path = "/" + http_path
access_token    = os.getenv("DATABRICKS_TOKEN")

# 从 http_path 尾部自动提取 cluster_id
# 格式：/sql/protocolv1/o/<workspace_id>/<cluster_id>
cluster_id = http_path.rstrip("/").split("/")[-1]

assert all([server_hostname, http_path, access_token, cluster_id]), "❌ .env 配置不完整"

print(f"Host:       {server_hostname}")
print(f"HTTP Path:  {http_path}")
print(f"Cluster ID: {cluster_id}")

In [ ]:
# -----------------------------------------------------------
# 3) 自动确保 cluster 处于 RUNNING 状态
# -----------------------------------------------------------
def ensure_cluster_running(timeout_seconds: int = 600, poll_interval: int = 15) -> str:
    """
    检查 cluster 状态；如果 TERMINATED，触发启动并轮询直至 RUNNING。
    返回最终状态。
    """
    base_url = f"https://{server_hostname}/api/2.0/clusters"
    headers  = {"Authorization": f"Bearer {access_token}"}

    def get_state() -> str:
        r = requests.get(
            f"{base_url}/get",
            headers=headers,
            params={"cluster_id": cluster_id},
            timeout=30,
        )
        r.raise_for_status()
        return r.json().get("state", "UNKNOWN")

    state = get_state()
    print(f"Current cluster state: {state}")

    if state == "RUNNING":
        return state

    # TERMINATING 时必须等它完全停下才能重启
    while state == "TERMINATING":
        print("Cluster is TERMINATING, waiting to fully stop...")
        time.sleep(poll_interval)
        state = get_state()

    if state == "TERMINATED":
        print("Starting cluster...")
        r = requests.post(
            f"{base_url}/start",
            headers=headers,
            json={"cluster_id": cluster_id},
            timeout=30,
        )
        # 409 = 已经在启动中
        if r.status_code not in (200, 409):
            r.raise_for_status()

    # 轮询直至 RUNNING
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        state = get_state()
        print(f"  ...state = {state}")
        if state == "RUNNING":
            print("✅ Cluster is RUNNING")
            return state
        if state in ("ERROR", "TERMINATED"):
            raise RuntimeError(f"Cluster failed to start, state={state}")
        time.sleep(poll_interval)

    raise TimeoutError(f"Cluster did not reach RUNNING within {timeout_seconds}s")


# -----------------------------------------------------------
# 4) 查询函数（自动唤醒 + 连接复用 + 断线重试）
# -----------------------------------------------------------
_conn = None

def _get_conn():
    global _conn
    if _conn is None:
        ensure_cluster_running()
        _conn = sql.connect(
            server_hostname=server_hostname,
            http_path=http_path,
            access_token=access_token,
        )
    return _conn

def query_databricks(sql_text: str) -> pd.DataFrame:
    global _conn
    try:
        with _get_conn().cursor() as cur:
            cur.execute(sql_text)
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
    except Exception:
        # 连接可能因集群重启失效，重置一次再试
        _conn = None
        with _get_conn().cursor() as cur:
            cur.execute(sql_text)
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

In [ ]:
# 冒烟测试：连接 + 查一条元数据
df_test = query_databricks("SELECT current_date() AS today, current_user() AS user")
df_test

## 2. Load Data

- `hc_md_df`：本地 master data CSV
- `shipment_df`：从 Databricks 拉当月 shipment

In [ ]:
DATA_DIR = PROJECT_ROOT / "Data"

hc_md_df = pd.read_csv(
    DATA_DIR / "hc_md_transformed.csv",
    encoding="cp1252",
    usecols=[
        "fpc", "product_name", "csu_ind", "rbu", "category", "brand",
        "level_1", "level_2", "level_3", "level_4", "level_5", "level_ni",
    ],
)
hc_md_df["fpc"] = hc_md_df["fpc"].astype(str).str.strip()

print(f"hc_md rows: {len(hc_md_df):,}")
hc_md_df.head()

## 3. Next Steps

在下面继续添加分析 cell（TG 计算 / MTD tracking / booklet merge 等）。